# Portfolio Optimization

**Prerequisites:** notebook **05** (portfolio construction and valuation).

This notebook demonstrates the configurable LP-based portfolio optimizer.
The single entry point `optimize_portfolio` accepts a JSON spec with a
flexible objective (maximize or minimize any metric expression),
attribute-based constraints, weight bounds, and turnover limits, then
solves for optimal weights and generates a trade list.

In [ ]:
import json
from datetime import date

import pandas as pd

from finstack.core.market_data import DiscountCurve, MarketContext
from finstack.portfolio import (
    optimize_portfolio,
    value_portfolio,
)

print("Imports OK")

## Market context

A single USD OIS discount curve used by all bonds in the portfolio.

In [ ]:
mc = MarketContext()
curve = DiscountCurve(
    "USD-OIS",
    date(2025, 1, 15),
    [
        (0.0, 1.0),
        (0.25, 0.9875),
        (0.5, 0.975),
        (1.0, 0.95),
        (2.0, 0.90),
        (5.0, 0.75),
        (10.0, 0.55),
    ],
    day_count="act_365f",
)
mc.insert(curve)
market_json = mc.to_json()
print("Market context ready")

## Portfolio: five fixed-rate bonds

Each position carries `attributes` — a map of key/value pairs where values
can be strings (e.g. `"rating": "AAA"`) or numbers (e.g. `"credit_score": 800.0`).
The optimizer uses these attributes for indicator-based constraints and
filtered metric expressions. Coupons range from 3% (AAA) to 8% (CCC),
reflecting typical credit spread tiers.

In [ ]:
def make_bond_position(pos_id, bond_id, coupon, maturity, rating, credit_score, notional=1_000_000):
    """Build a portfolio position dict for a fixed-rate bond."""
    return {
        "position_id": pos_id,
        "entity_id": "FUND",
        "instrument_id": bond_id,
        "instrument_spec": {
            "type": "bond",
            "spec": {
                "id": bond_id,
                "notional": {"amount": str(notional), "currency": "USD"},
                "issue_date": "2024-01-15",
                "maturity": maturity,
                "discount_curve_id": "USD-OIS",
                "cashflow_spec": {
                    "Fixed": {
                        "coupon_type": "Cash",
                        "rate": coupon,
                        "freq": {"count": 6, "unit": "months"},
                        "dc": "Thirty360",
                        "bdc": "following",
                        "calendar_id": "weekends_only",
                        "end_of_month": False,
                        "payment_lag_days": 0,
                        "stub": "None",
                    }
                },
                "credit_curve_id": None,
                "call_put": None,
                "attributes": {"tags": [], "meta": {}},
                "pricing_overrides": {},
            },
        },
        "quantity": 1.0,
        "unit": "units",
        "attributes": {
            "rating": rating,
            "sector": "corporate",
            "credit_score": credit_score,
        },
    }


positions = [
    make_bond_position("POS-AAA", "BOND-AAA-5Y", 0.03, "2029-01-15", "AAA", 800.0),
    make_bond_position("POS-AA",  "BOND-AA-3Y",  0.04, "2027-01-15", "AA", 750.0),
    make_bond_position("POS-BBB", "BOND-BBB-5Y", 0.055, "2029-01-15", "BBB", 650.0),
    make_bond_position("POS-BB",  "BOND-BB-7Y",  0.07, "2031-01-15", "BB", 550.0),
    make_bond_position("POS-CCC", "BOND-CCC-5Y", 0.08, "2029-01-15", "CCC", 400.0),
]

portfolio_spec = json.dumps({
    "id": "bond-fund",
    "as_of": "2025-01-15",
    "base_ccy": "USD",
    "entities": {"FUND": {"id": "FUND"}},
    "positions": positions,
})

print(f"Portfolio: {len(positions)} bonds")
for p in positions:
    print(f"  {p['position_id']:10s}  coupon={p['instrument_spec']['spec']['cashflow_spec']['Fixed']['rate']:.1%}  rating={p['attributes']['rating']}  score={p['attributes']['credit_score']}")

## Baseline valuation

Value the portfolio before optimization to see the starting state.

In [ ]:
val = json.loads(value_portfolio(portfolio_spec, market_json))

rows = []
for pos_id, pv in val.get("position_values", {}).items():
    rows.append({
        "position": pv["position_id"],
        "pv": float(pv["value_base"]["amount"]),
        "currency": pv["value_base"]["currency"],
    })

df_val = pd.DataFrame(rows)
total = val["total_base_ccy"]
print(f"Total portfolio value: {float(total['amount']):,.2f} {total['currency']}")
df_val

## Example 1 — Basic yield maximization with rating cap

Maximize value-weighted YTM while constraining CCC exposure to at most 10%
of portfolio weight. The `MetricBound` constraint uses an `AttributeIndicator`
to select positions where `rating == "CCC"`, then bounds the weighted sum of
that 0/1 indicator.

The optimizer uses an LP formulation: weights are value-based shares that
sum to 1.0 (fully invested budget constraint).

In [ ]:
opt_spec = json.dumps({
    "portfolio": json.loads(portfolio_spec),
    "objective": {
        "Maximize": {
            "ValueWeightedAverage": {
                "metric": {"Metric": "Ytm"}
            }
        }
    },
    "constraints": [
        {"MetricBound": {
            "label": "ccc_cap",
            "metric": {"WeightedSum": {
                "metric": {"AttributeIndicator": {
                    "key": "rating",
                    "op": "Eq",
                    "value": "CCC"
                }}
            }},
            "op": "Le",
            "rhs": 0.10,
        }},
        {"WeightBounds": {
            "label": "position_limits",
            "filter": "All",
            "min": 0.0,
            "max": 0.40,
        }},
    ],
    "weighting": "ValueWeight",
    "missing_metric_policy": "Zero",
    "label": "basic_yield_max",
})

result = json.loads(optimize_portfolio(opt_spec, market_json))
print(f"Status:    {result['status_label']}")
print(f"Feasible:  {result['is_feasible']}")
print(f"Objective: {result['objective_value']:.4%}")

## Example 2 — Multi-constraint optimization

Add more constraints to the optimization:
- CCC weight ≤ 10% (`MetricBound` with `AttributeIndicator`)
- BB weight ≤ 30% (`MetricBound` with `AttributeIndicator`)
- Max turnover ≤ 40% (`MaxTurnover`)
- Each position weight in [0%, 40%] (`WeightBounds`)

In [ ]:
opt_spec = json.dumps({
    "portfolio": json.loads(portfolio_spec),
    "objective": {
        "Maximize": {
            "ValueWeightedAverage": {
                "metric": {"Metric": "Ytm"}
            }
        }
    },
    "constraints": [
        {"MetricBound": {
            "label": "ccc_cap",
            "metric": {"WeightedSum": {
                "metric": {"AttributeIndicator": {
                    "key": "rating",
                    "op": "Eq",
                    "value": "CCC"
                }}
            }},
            "op": "Le",
            "rhs": 0.10,
        }},
        {"MetricBound": {
            "label": "bb_cap",
            "metric": {"WeightedSum": {
                "metric": {"AttributeIndicator": {
                    "key": "rating",
                    "op": "Eq",
                    "value": "BB"
                }}
            }},
            "op": "Le",
            "rhs": 0.30,
        }},
        {"MaxTurnover": {
            "label": "turnover_limit",
            "max_turnover": 0.40,
        }},
        {"WeightBounds": {
            "label": "position_limits",
            "filter": "All",
            "min": 0.0,
            "max": 0.40,
        }},
    ],
    "weighting": "ValueWeight",
    "missing_metric_policy": "Zero",
    "label": "max_yield_constrained",
})

result = json.loads(optimize_portfolio(opt_spec, market_json))

print(f"Status:    {result['status_label']}")
print(f"Feasible:  {result['is_feasible']}")
print(f"Objective: {result['objective_value']:.4%}")
print(f"Turnover:  {result['turnover']:.2%}")
print(f"Binding:   {result['binding_constraints']}")

### Optimal weights and deltas

In [ ]:
weights_df = pd.DataFrame([
    {
        "position": pos_id,
        "current_wt": result["current_weights"].get(pos_id, 0),
        "optimal_wt": wt,
        "delta": result["weight_deltas"].get(pos_id, 0),
    }
    for pos_id, wt in result["optimal_weights"].items()
])
weights_df.style.format({"current_wt": "{:.2%}", "optimal_wt": "{:.2%}", "delta": "{:+.2%}"})

### Trade list

The optimizer returns a sorted trade list with direction, quantities, and
weight changes.

In [ ]:
if result["trades"]:
    trades_df = pd.DataFrame(result["trades"])
    display(trades_df)
else:
    print("No trades needed (portfolio already optimal).")

## Example 3 — Filtered metrics

`MetricExpr` variants accept an optional `filter` field that scopes the
aggregation to a subset of positions. This enables constraints like
"the combined weight of BB and CCC positions must be at least 20%"
without affecting other positions.

Filters compose with `And`, `Or`, `Not`, `ByAttribute`, `ByEntityId`,
and `ByPositionIds`.

In [ ]:
opt_spec = json.dumps({
    "portfolio": json.loads(portfolio_spec),
    "objective": {
        "Maximize": {
            "ValueWeightedAverage": {
                "metric": {"Metric": "Ytm"}
            }
        }
    },
    "constraints": [
        {"MetricBound": {
            "label": "ccc_cap",
            "metric": {"WeightedSum": {
                "metric": {"AttributeIndicator": {
                    "key": "rating",
                    "op": "Eq",
                    "value": "CCC"
                }}
            }},
            "op": "Le",
            "rhs": 0.15,
        }},
        {"MetricBound": {
            "label": "hy_weight_floor",
            "metric": {"WeightedSum": {
                "metric": {"Constant": 1.0},
                "filter": {"Or": [
                    {"ByAttribute": {"key": "rating", "op": "Eq", "value": "BB"}},
                    {"ByAttribute": {"key": "rating", "op": "Eq", "value": "CCC"}},
                ]}
            }},
            "op": "Ge",
            "rhs": 0.20,
        }},
        {"WeightBounds": {
            "label": "position_limits",
            "filter": "All",
            "min": 0.0,
            "max": 0.50,
        }},
    ],
    "weighting": "ValueWeight",
    "missing_metric_policy": "Zero",
    "label": "filtered_metrics_demo",
})

result3 = json.loads(optimize_portfolio(opt_spec, market_json))
print(f"Status:    {result3['status_label']}")
print(f"Feasible:  {result3['is_feasible']}")
print(f"Objective: {result3['objective_value']:.4%}")

## Summary

| Component | Description |
|---|---|
| `optimize_portfolio` | Single entry point for all LP optimization |
| `MetricBound` | Bound any metric expression (replaces old tag-based constraints) |
| `WeightBounds` | Min/max weight per position, with optional filter |
| `MaxTurnover` | Portfolio turnover limit |
| `Budget` | Weight normalization (sum to 1.0) |
| `AttributeIndicator` | 0/1 indicator for attribute matching |
| `PositionFilter` | Scope metrics to position subsets (`ByAttribute`, `Or`, `Not`, …) |

The optimizer runs entirely in Rust and returns JSON results including
optimal weights, trade lists, and binding constraint diagnostics. All
constraint types are composable and can be mixed freely.